In [1]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.8 MB/s eta 0:00:00


In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from huggingface_hub import login
import evaluate

In [3]:

class DatasetHandler:
    def __init__(self, dataset_name, sample_size=1000):
        self.dataset_name = dataset_name
        self.sample_size = sample_size

    def load_and_prepare(self):
        dataset = load_dataset(self.dataset_name, split="train")
        dataset = dataset.shuffle(seed=42).select(range(self.sample_size))

        def format_instruction(example):
            return {
                "text": f"User: {example['instruction']}\nAssistant: {example['response']}"
            }

        formatted_dataset = dataset.map(format_instruction)

        split_dataset = formatted_dataset.train_test_split(test_size=0.1, seed=42)

        self.train_data = split_dataset["train"]
        self.test_data = split_dataset["test"]

        return self.train_data, self.test_data


In [4]:
class ModelHandler:
    def __init__(self, model_id):
        self.model_id = model_id

    def setup_tokenizer(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.tokenizer.padding_side = "right"
        return self.tokenizer

    def setup_model(self):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            quantization_config=bnb_config,
            device_map="auto"
        )

        self.model = prepare_model_for_kbit_training(self.model)

        return self.model

    def apply_lora(self):
        peft_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )

        self.model = get_peft_model(self.model, peft_config)
        return self.model



In [5]:
class TrainerHandler:
    def __init__(self, model, train_data):
        self.model = model
        self.train_data = train_data

    def setup_training_args(self):
        self.training_args = TrainingArguments(
            output_dir="./results",
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            optim="paged_adamw_32bit",
            save_steps=50,
            logging_steps=10,
            learning_rate=2e-4,
            weight_decay=0.001,
            bf16=True,
            max_grad_norm=0.3,
            max_steps=200,
            warmup_ratio=0.03,
            lr_scheduler_type="cosine"
        )
        return self.training_args

    def train(self):
        self.trainer = SFTTrainer(
            model=self.model,
            train_dataset=self.train_data,
            args=self.training_args,
        )

        print("🚀 Starting Fine-Tuning...")
        self.trainer.train()

    def save_model(self):
        self.trainer.model.save_pretrained("fine_tuned_model")






In [6]:
class Evaluator:
    def __init__(self, model, tokenizer, test_data):
        self.model = model
        self.tokenizer = tokenizer
        self.test_data = test_data

    def generate_response(self, prompt):
        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=100
        )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def evaluate_model(self):
        rouge = evaluate.load("rouge")

        predictions = []
        references = []

        for i in range(20):
            prompt = self.test_data[i]["instruction"]
            expected = self.test_data[i]["response"]

            inputs = self.tokenizer(
                f"User: {prompt}\nAssistant:",
                return_tensors="pt"
            ).to("cuda")

            outputs = self.model.generate(**inputs, max_new_tokens=50)
            pred = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

            pred = pred.split("Assistant:")[-1].strip()

            predictions.append(pred)
            references.append(expected)

        results = rouge.compute(predictions=predictions, references=references)

        print("📊 Evaluation Results:", results)



In [ ]:
class Pipeline:
    def __init__(self):
        self.model_id = "Qwen/Qwen2.5-1.5B-Instruct"

    def run(self):

        login(token="")

        # Dataset
        dataset_handler = DatasetHandler(
            "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
        )
        train_data, test_data = dataset_handler.load_and_prepare()

        # Model
        model_handler = ModelHandler(self.model_id)
        tokenizer = model_handler.setup_tokenizer()
        model = model_handler.setup_model()
        model = model_handler.apply_lora()

        # Training
        trainer = TrainerHandler(model, train_data)
        trainer.setup_training_args()
        trainer.train()
        trainer.save_model()

        # Evaluation
        evaluator = Evaluator(model, tokenizer, test_data)
        evaluator.evaluate_model()


pipeline = Pipeline()
pipeline.run()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.792293
20,1.557026
30,1.400688
40,1.243492
50,1.237229
60,1.169323
70,1.157217
80,1.140674
90,1.069017
100,1.102813


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


📊 Evaluation Results: {'rouge1': np.float64(0.01959856697507026), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.019519679089674953), 'rougeLsum': np.float64(0.018976842609494594)}
